In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM



model_name = "gpt2-medium"
model=AutoModelForCausalLM.from_pretrained(model_name)
tokenise=AutoTokenizer.from_pretrained(model_name)
model.eval()
device=torch.device('cuda'if torch.cuda.is_available() else'cpu')
model.to(device)


prompt = f"Question:{'How do I make a...?'} \nAnswer:"

inputs=tokenise(prompt,return_tensors='pt').to(model.device)
input_ids=inputs["input_ids"].to(model.device)
with torch.no_grad():
  outputs=model(**inputs)
# extract logits at last position
logits=outputs.logits
next_logits=logits[:,-1,:]

prob= F.softmax(next_logits,dim=-1)
top_k= torch.topk(prob,dim=-1,k=50)
indices=top_k.indices.squeeze(0)
values=top_k.values.squeeze(0)
print(top_k.indices.shape)
for i in range(50):
    top_k_ids=[]
    token=tokenise.decode([indices[i].item()])
    proba=values[i].item()
    top_k_ids.append(
        {
        "Token":token,
        "Probability":proba,
        }
    )
print(top_k_ids[0])

from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained("gpt2")

def separate_harmful(topk_ids, topk_probs, harmful_ids):
    # TODO: split topk into harmful vs benign
    pass

def separation_score(harmful_probs, benign_probs):
    # TODO: mean difference
    pass

def ablate_layer(model, prompt, layer_idx):
    # TODO: use model.run_with_hooks() to zero out layer_idx
    pass

def run(prompt, harmful_tokens, k=50):
    tokens = model.to_tokens(prompt)
    harmful_ids = [model.to_single_token(t) for t in harmful_tokens]

    logits, cache = model.run_with_cache(tokens)
    probs = torch.softmax(logits[0, -1, :], dim=-1)

    topk_ids, topk_probs = get_t
    harmful_probs, benign_probs = separate_harmful(topk_ids, topk_probs, harmful_ids)
    score = separation_score(harmful_probs, benign_probs)

    # TODO: loop over layers, ablate each, recompute score, compare
    
    return score, cache

if __name__ == "__main__":
    prompt = "TODO"
    harmful_tokens = ["TODO"]
    score, cache = run(prompt, harmful_tokens)"""

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 50])
{'Token': 'P', 'Probability': 0.002818241249769926}


'implentation of TransformerLens\nfrom transformer_lens import HookedTransformer\n\nmodel = HookedTransformer.from_pretrained("gpt2")\n\ndef get_topk_tokens(probs, k=50):\n    topk = torch.topk(probs, k=k)\n    return topk.indices, topk.values\n\ndef separate_harmful(topk_ids, topk_probs, harmful_ids):\n    # TODO: split topk into harmful vs benign\n    pass\n\ndef separation_score(harmful_probs, benign_probs):\n    # TODO: mean difference\n    pass\n\ndef ablate_layer(model, prompt, layer_idx):\n    # TODO: use model.run_with_hooks() to zero out layer_idx\n    pass\n\ndef run(prompt, harmful_tokens, k=50):\n    tokens = model.to_tokens(prompt)\n    harmful_ids = [model.to_single_token(t) for t in harmful_tokens]\n\n    logits, cache = model.run_with_cache(tokens)\n    probs = torch.softmax(logits[0, -1, :], dim=-1)\n\n    topk_ids, topk_probs = get_topk_tokens(probs, k)\n    harmful_probs, benign_probs = separate_harmful(topk_ids, topk_probs, harmful_ids)\n    score = separation_score